In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! pip install langchain-gigachat langchain-community sentence-transformers langchain-huggingface chromadb langchain-chroma

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-chroma to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking

## Базовый пример работы с API

In [ ]:
with open('/content/drive/MyDrive/gigachat_api_key.txt', 'r') as f:
  api_key = f.read()

In [ ]:
from langchain_gigachat.chat_models.gigachat import GigaChat

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
model = GigaChat(
    #model="GigaChat-Pro",
    model="GigaChat-Max",
    credentials=api_key.strip(),
    verify_ssl_certs=False,
)

In [ ]:
messages = [
    SystemMessage(
        content="Ты специалист по машинному обучению и цифровым гуманитарным наукам. Ты помогаешь пользователям, отвечая на их вопросы."
    )
]

while(True):
    user_input = input("Пользователь: ")
    if user_input == "пока":
      break
    messages.append(HumanMessage(content=user_input))
    res = model.invoke(messages)
    messages.append(res)
    print("GigaChat: ", res.content)

Пользователь: что такое RAG?
GigaChat:  RAG (Retrieval-Augmented Generation) — это подход в области искусственного интеллекта для генерации текста с использованием информации из внешних источников данных.

В традиционной модели генеративного ИИ (например, GPT-3), модель обучается на больших объёмах текстов и затем самостоятельно генерирует ответы без доступа к внешним источникам знаний. В результате ответ может быть неточным или устаревшим, если данные изменились после завершения обучения.

С другой стороны, метод RAG сочетает две идеи:
1. **Поисковая система** (retrieval): Модель сначала извлекает релевантную информацию из базы данных или поисковой системы, используя методы поиска информации (Information Retrieval).
2. **Генерация ответа**: Затем эта информация используется для уточнения и улучшения ответов генеративной модели.

Таким образом, подход RAG позволяет моделям использовать актуальную информацию при генерации ответов, повышая точность и полезность результатов.

Основные пре

## Retrieval Augmented Generation (RAG)

simple-rag.svg

Картинка: https://learn.microsoft.com/ru-ru/azure/databricks/generative-ai/retrieval-augmented-generation

In [ ]:
question = "Что главный герой рассказа \"Скучная история\" Чехова думает про сад в университете, в котором работает?"

In [ ]:
messages = [
    SystemMessage(
        content="Ты - специалист по русской литературе 19-20 веков."
    ),
    HumanMessage(content=question),
]

print(model.invoke(messages).content)

Главный герой рассказа Антона Павловича Чехова «Скучная история» — Николай Степанович, профессор медицины, размышляет о саде университета следующим образом:

> «А теперь взгляните на наш университетский сад: узкие дорожки его протоптаны студенческими ногами и представляют собою тропинки, по которым прошли тысячи ног... сколько шагов сделано по этим дорожкам! Сколько надежд, мечтаний, стремлений прошло здесь за эти годы!..»

Николай Степанович задумывается над тем, что этот скромный, маленький сад является свидетелем множества судеб студентов, которые через него проходили, испытывали волнения, радость открытий, надежды и разочарования.

Таким образом, сад для профессора становится символом жизни, времени и человеческих чувств, проходящих сквозь поколения. Он видит в нём отражение своей собственной судьбы и судьбы других людей, наполнявших свою жизнь трудом, мыслями и эмоциями.


In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
)

loader = TextLoader("/content/drive/MyDrive/ML_training_data/скучная история.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # влияет на качество ответов!
    chunk_overlap=200,
)
documents = text_splitter.split_documents(documents)
print(f"Total documents: {len(documents)}")

Total documents: 85


In [ ]:
documents[:5]

[Document(metadata={'source': '/content/drive/MyDrive/ML_training_data/скучная история.txt'}, page_content='\ufeffСкучная история\nАнтон Павлович Чехов\n\n\n\n\n\nСкучная история\n\n\n\nИз записок старого человека\n\n\nI'),
 Document(metadata={'source': '/content/drive/MyDrive/ML_training_data/скучная история.txt'}, page_content='Есть в России заслуженный профессор Николай Степанович такой-то, тайный советник и кавалер; у\xa0него так много русских и иностранных орденов, что когда ему приходится надевать их, то студенты величают его иконостасом. Знакомство у него самое аристократическое; по крайней мере, за последние двадцать пять – тридцать лет в России нет и не было такого знаменитого ученого, с которым он не был бы коротко знаком. Теперь дружить ему не с кем, но если говорить о прошлом, то длинный список его славных'),
 Document(metadata={'source': '/content/drive/MyDrive/ML_training_data/скучная история.txt'}, page_content='пять – тридцать лет в России нет и не было такого знаменито

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/distiluse-base-multilingual-cased-v2"
model_kwargs = {'device': 'cpu'} # or cuda
encode_kwargs = {'normalize_embeddings': False}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from chromadb.config import Settings
from langchain_chroma import Chroma

In [ ]:
db = Chroma.from_documents(
    documents,
    hf,
    client_settings=Settings(anonymized_telemetry=False),
)

In [ ]:
docs = db.similarity_search(question, k=4)
len(docs)

4

In [ ]:
docs

[Document(id='645cecd4-ea9d-4a43-ae2f-fa89f9db1efd', metadata={'source': '/content/drive/MyDrive/ML_training_data/скучная история.txt'}, page_content='\ufeffСкучная история\nАнтон Павлович Чехов\n\n\n\n\n\nСкучная история\n\n\n\nИз записок старого человека\n\n\nI'),
 Document(id='9c106956-f0a8-427c-b6aa-54205d975404', metadata={'source': '/content/drive/MyDrive/ML_training_data/скучная история.txt'}, page_content='наш сад. С тех пор как я был студентом, он, кажется, не стал ни лучше, ни хуже. Я его не люблю. Было бы гораздо умнее, если бы вместо чахоточных лип, желтой акации и редкой стриженой сирени росли тут высокие сосны и хорошие дубы. Студент, настроение которого в большинстве создается обстановкой, на каждом шагу, там, где он учится, должен видеть перед собою только высокое, сильное и изящное… Храни его бог от тощих деревьев, разбитых окон, серых стен и дверей, сбитых рваной клеенкой.'),
 Document(id='290916ac-0fca-402c-ace8-60829ade5d2e', metadata={'source': '/content/drive/MyDr

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "Ты - специалист по русской литературе 19-20 веков."
    "Ты должен ответить на вопрос пользователя с использованием данных из рассказа.\n"
    "Отвечай коротко, не более 2-3 предложений.\n"
    "Вот контекст для ответа:"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

retriever = db.as_retriever()

question_answer_chain = create_stuff_documents_chain(model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
rag_chain.invoke({"input": question})

{'input': 'Что главный герой рассказа "Скучная история" Чехова думает про сад в университете, в котором работает?',
 'context': [Document(id='048b32f2-61e3-4f80-bfda-bda93760ee36', metadata={'source': '/content/drive/MyDrive/ML_training_data/скучная история.txt'}, page_content='\ufeffСкучная история\nАнтон Павлович Чехов\n\n\n\n\n\nСкучная история\n\n\n\nИз записок старого человека\n\n\nI'),
  Document(id='244e0977-f7fb-47b5-b0ae-4ec53c997d26', metadata={'source': '/content/drive/MyDrive/ML_training_data/скучная история.txt'}, page_content='наш сад. С тех пор как я был студентом, он, кажется, не стал ни лучше, ни хуже. Я его не люблю. Было бы гораздо умнее, если бы вместо чахоточных лип, желтой акации и редкой стриженой сирени росли тут высокие сосны и хорошие дубы. Студент, настроение которого в большинстве создается обстановкой, на каждом шагу, там, где он учится, должен видеть перед собою только высокое, сильное и изящное… Храни его бог от тощих деревьев, разбитых окон, серых стен и

In [ ]:
rag = rag_chain.invoke({"input": question})["answer"]
print(f"RAG: {rag}")

RAG: Главный герой не любит университетский сад, считая, что вместо чахоточных лип, жёлтой акации и редкой стриженой сирени должны расти высокие сосны и красивые дубы, чтобы студенты видели вокруг себя только высокое, сильное и изящное.


## Структурированная выдача (GigaChat-2-Max)

Как заставить нейросеть выдавать данные в заданном формате?

Входные данные: список существительных.

Задача: для каждого слова определить, является оно конкретным или абстрактным.

Желаемый формат выдачи: [{"word": 'word', "category": 'abstract/concrete', "confidence": 0.0}, {"word": 'word1', "category": 'abstract/concrete', "confidence": 1.0}...]

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class WordClass(BaseModel):
  """Слово, его разряд по значению: конкретное или абстрактное, а также оценка уверенности модели"""
  word: str = Field(..., description="слово")
  category: str = Field(..., description="разряд слова по значению: конкретное или абстрактное")
  confidence: float = Field(..., description="уверенность в оценке категории слова от 0 до 1")

In [ ]:
class WordCategoryListWrapper(BaseModel):
  """Список слов с разрядами по значению и оценками модели"""
  items: List[WordClass]

In [ ]:
structured_llm = model.with_structured_output(WordCategoryListWrapper)

In [ ]:
prompt = """
Ты - эксперт по современному русскому языку. Перед тобой список существительных. Оцени каждое из них на предмет того, является оно конкретным или абстрактным.

Конкретные существительные называют чувственно воспринимаемые предметы — вещи (стол), лица (Марина), которые можно воспринять зрением и осязанием.

Абстрактные существительные обозначают отвлеченные понятия (радость), признаки (белизна), действия (рисование).

Присвой каждому слову разряд по значению: конкретное или абстрактное. Каждый раз, когда присваиваешь слову разряд, оцени, насколько ты уверен в своем решении, по шкале от 0 до 1: 0 - совсем не уверен, 1 - полностью уверен.

Список слов:

"""

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/ML_training_data/список абстрактных слов - встретилось в учебниках .csv')

In [ ]:
df[:10]

,слово,частотность во фрагментах
0,время,1411
1,часть,1021
2,жизнь,979
3,год,953
4,война,941
5,власть,894
6,сила,850
7,развитие,780
8,мир,744
9,начало,707


In [ ]:
word_list = ', '.join(df['слово'].tolist()[:10])

In [ ]:
text_input = prompt + word_list

In [ ]:
out = structured_llm.invoke(text_input)

In [ ]:
# здесь "model" - это наша pydantic-модель WordCategoryListWrapper
results = out.model_dump_json()

In [ ]:
results

'{"items":[{"word":"время","category":"абстрактное","confidence":1.0},{"word":"часть","category":"конкретное","confidence":0.9},{"word":"жизнь","category":"абстрактное","confidence":1.0},{"word":"год","category":"конкретное","confidence":0.9},{"word":"война","category":"конкретное","confidence":0.9},{"word":"власть","category":"абстрактное","confidence":1.0},{"word":"сила","category":"абстрактное","confidence":1.0},{"word":"развитие","category":"абстрактное","confidence":1.0},{"word":"мир","category":"абстрактное","confidence":0.8},{"word":"начало","category":"абстрактное","confidence":1.0}]}'

# Agentic RAG

Напишем ИИ-агента, который будет уметь отвечать на вопросы по рассказам Чехова, пользуясь RAG.

Агент будет состоять из следующих нод:


1.   Нода проверки релевантности вопроса - агент должен игнорировать вопросы, не связанные с его "специальностью";
2.   Нода вызова RAG;
3.   Нода выдачи финального ответа.



## Добавление новых текстов в базу

In [ ]:
import os

In [ ]:
os.listdir("/content/drive/MyDrive/ML_training_data/more_Chekhov")

['Дом с мезонином.txt',
 'Анна на шее.txt',
 'Толстый и тонкий.txt',
 'Человек в футляре.txt',
 'Дама с собачкой.txt']

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
loader = DirectoryLoader(
    "/content/drive/MyDrive/ML_training_data/more_Chekhov",
    glob="**/*.txt",
    loader_cls=TextLoader,
    silent_errors=True,               # skip files that fail to load (e.g. encoding issues)
    recursive=True                    # include subdirectories
)
additional_documents = loader.load()

In [ ]:
additional_documents = text_splitter.split_documents(additional_documents)
additional_documents[:5]

[Document(metadata={'source': '/content/drive/MyDrive/ML_training_data/more_Chekhov/Дом с мезонином.txt'}, page_content='Дом с мезонином'),
 Document(metadata={'source': '/content/drive/MyDrive/ML_training_data/more_Chekhov/Дом с мезонином.txt'}, page_content='Рассказ художника\nI'),
 Document(metadata={'source': '/content/drive/MyDrive/ML_training_data/more_Chekhov/Дом с мезонином.txt'}, page_content='Это было шесть-семь лет тому назад, когда я жил в одном из уездов Т-ой губернии, в имении помещика Белокурова, молодого человека, который вставал очень рано, ходил в поддевке, по вечерам пил пиво и все жаловался мне, что он нигде и ни в ком не встречает сочувствия. Он жил в саду во флигеле, а я в старом барском доме, в громадной зале с колоннами, где не было никакой мебели, кроме широкого дивана, на котором я спал, да еще стола, на котором я раскладывал пасьянс. Тут всегда, даже в тихую погоду,'),
 Document(metadata={'source': '/content/drive/MyDrive/ML_training_data/more_Chekhov/Дом с м

In [ ]:
db.add_documents(additional_documents)

['2529af67-a3d8-4a3c-8757-1acbdead72d9',
 '700050fe-de25-4016-a991-53f2088931fd',
 'de57d9eb-857e-4d42-8bc4-9fd80152f496',
 '03ce1aa6-080e-46db-a181-45cb53814ef0',
 '6807efe9-ea3c-46b7-b41b-470a3af2702d',
 '3e404548-11c8-401a-9f94-e1a1bdd2eb2a',
 'fcb8bedd-38d5-40fa-b369-6c6ee39e2246',
 'bb7e31c8-0abd-4612-baf5-f16810ee7cce',
 'bb65922c-bc95-49ff-94a5-c9c64e30d587',
 '304b8db0-ed79-4727-9de9-9a2537a60697',
 'd26fad71-21a8-40f7-8e2f-40511f7fbdf6',
 '8d3db62a-2fd6-4d5f-a977-0710d4e7e5f6',
 '6a46fd7e-d109-4add-9f16-ca2a2dffae6d',
 'f51ae1f8-29a0-4e85-acc6-b8f65b371465',
 'cd369782-b1f9-4819-b006-dae55a3b1998',
 '24ce87c0-c918-408c-b26d-81f3386370fa',
 'c7e507d6-3839-4d76-a62c-2dacbcc0e22f',
 '980f091b-dad9-456d-8fab-eaf691a75317',
 'b1b7cea8-aab0-4694-b2a7-df21a01d3aae',
 '16d9ca3a-775e-488c-bdc8-44b8c1367457',
 '894adc94-89e8-4100-85b0-d5011304a26d',
 '8933d527-e1e8-4a60-887c-ea075e529662',
 '4a1cf1a4-3898-402b-9fec-f7ad3a7e3bf4',
 '32e80df8-7a3b-4cc2-815f-7374cd00e320',
 '45eb4553-6357-

Проверим, что документы добавились:

In [ ]:
rag_chain.invoke({"input": "Назови полное имя Вареньки, которую хотели выдать замуж за Беликова"})

{'input': 'Назови полное имя Вареньки, которую хотели выдать замуж за Беликова',
 'context': [Document(id='aa7848a4-5295-4f0f-8797-483d0e50c650', metadata={'source': '/content/drive/MyDrive/ML_training_data/more_Chekhov/Человек в футляре.txt'}, page_content='Чего только не делается у нас в провинции от скуки, сколько ненужного, вздорного! И это потому, что совсем не делается то, что нужно. Ну вот к чему нам вдруг понадобилось женить этого Беликова, которого даже и вообразить нельзя было женатым? Директорша, инспекторша и все наши гимназические дамы ожили, даже похорошели, точно вдруг увидели цель жизни. Директорша берет в театре ложу, и смотрим — в ее ложе сидит Варенька с этаким веером, сияющая, счастливая, и рядом с ней Беликов, маленький,'),
  Document(id='c5e55ab8-0473-43da-8748-adfef80c14d1', metadata={'source': '/content/drive/MyDrive/ML_training_data/more_Chekhov/Человек в футляре.txt'}, page_content='пригласил и Беликова и Вареньку. Одним словом, заработала машина. Оказалось, ч

## Создание агента

Мы представим агента в виде графа, в котором каждая нода будет выполнять свою функцию.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
# Состояние, которое будет передаваться от одной ноды агента к другой
class State(TypedDict):
  question: str
  relevant: Optional[bool]
  answer: Optional[str]

In [ ]:
class CheckSystemOutput(BaseModel):
  """Определение релевантности вопроса"""
  relevance: bool = Field(..., description="Релевантность вопроса: true, если вопрос относится к рассказам Чехова, false, если не относится")


check_relevance_system_prompt = """
Ты - специалист по классификации сообщений.
Ты - часть системы, которая умеет отвечать ТОЛЬКО на вопросы о рассказах А.П. Чехова.
"""


def get_check_system_human_prompt(question):
 return f"""
 Внимательно посмотри на этот вопрос: {question}
 Относится ли он к рассказам Чехова? Верни True, если относится, и False, если не относится.
 """


def check_relevance_node(state: State) -> State:
  """
  Нода проверки релевантности вопроса.
  Агент умеет отвечать только на вопросы по рассказам Чехова.
  """
  question = state["question"]
  human_prompt = get_check_system_human_prompt(question)

  messages = [
  SystemMessage(content=check_relevance_system_prompt),
  HumanMessage(content=human_prompt),
  ]

  structured_llm = model.with_structured_output(CheckSystemOutput)

  try:
    output = structured_llm.invoke(messages)
  except:
    output = CheckSystemOutput(relevance=False)

  return {**state, "relevant": output.relevance}

Вариант ноды check_relevance без использования .with_structured_output():

In [ ]:
# @title
class CheckSystemOutput(BaseModel):
  """Определение релевантности вопроса"""
  relevance: bool = Field(..., description="Релевантность вопроса: true, если вопрос относится к рассказам Чехова, false, если не относится")


check_relevance_system_prompt = """
Ты - специалист по классификации сообщений.
Ты - часть системы, которая умеет отвечать ТОЛЬКО на вопросы о рассказах А.П. Чехова.
"""


def get_check_system_human_prompt(question):
 return f"""
 Внимательно посмотри на этот вопрос: {question}
 Относится ли он к рассказам Чехова? Верни True, если относится, и False, если не относится.
 Верни ТОЛЬКО JSON:
 {{
  "relevance": true, если вопрос относится к рассказам Чехова, false, если не относится
 }}
 """


def check_relevance_node_no_structured_output(state: State) -> State:
  """
  Нода проверки релевантности вопроса.
  Агент умеет отвечать только на вопросы по рассказам Чехова.
  """
  question = state["question"]
  human_prompt = get_check_system_human_prompt(question)

  messages = [
  SystemMessage(content=check_relevance_system_prompt),
  HumanMessage(content=human_prompt),
  ]

  response = model.invoke(messages)

  try:
    output = CheckSystemOutput.model_validate_json(response.content)
  except:
    output = CheckSystemOutput(relevance=False)

  return {**state, "relevant": output.relevance}

In [ ]:
# функция-роутер для графа
# определяет, в какую ноду пойдет граф в зависимости от ответа ноды check_relevance_node
def choose_next(state):
  return "rag" if state.get("relevant") else "respond"

In [ ]:
def call_rag_node(state: State) -> State:
  """
  Нода вызова RAG.
  """
  question = state["question"]

  try:
    output = rag_chain.invoke({"input": question})["answer"]
  except:
    output = "Произошла ошибка при вызове RAG."

  return {**state, "answer": output}

In [ ]:
def response_node(state: State):
  """
  Нода финального ответа
  """
  if state["relevant"]:
      return {"answer": state["answer"]}
  else:
      return {"answer": "Я умею отвечать только на вопросы по рассказам А.П. Чехова"}

In [ ]:
graph = StateGraph(State)

graph.add_node("check_relevance", check_relevance_node)
graph.add_node("rag", call_rag_node)
graph.add_node("respond", response_node)

graph.set_entry_point("check_relevance")

graph.add_conditional_edges("check_relevance",
                            choose_next,
                             {"rag": "rag", "respond": "respond"})

graph.add_edge("rag", "respond")
graph.add_edge("respond", END)

agent = graph.compile()

In [ ]:
state = State(question="Какая сегодня погода?")
agent.invoke(state)

{'question': 'Какая сегодня погода?',
 'relevant': False,
 'answer': 'Я умею отвечать только на вопросы по рассказам А.П. Чехова'}

In [ ]:
state = State(question="Кем приходятся друг другу Гуров и Анна Сергеевна в конце рассказа?")
agent.invoke(state)

{'question': 'Кем приходятся друг другу Гуров и Анна Сергеевна в конце рассказа?',
 'relevant': True,
 'answer': 'Гуров и Анна Сергеевна становятся любовниками, тайно встречаясь и сохраняя свои отношения в секрете от окружающих.'}